In [6]:
# Risk Classification Reclassification Based on Sand Dunes and Artificial Structures
# This Colab notebook reads risk rank shapefiles and reclassifies them based on sand dunes and artificial structures

# Install required libraries
!pip install geopandas shapely fiona

# Import libraries
import geopandas as gpd
from google.colab import drive
import os
from shapely.ops import unary_union
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Define paths to your shapefiles (Update these with your actual Google Drive paths)
# Example: '/content/drive/My Drive/your_folder/risk_rank.shp'
RISK_SHAPEFILE_PATH = '/content/drive/My Drive/AP_Inputs/outputs/inundation_risk_areas_detailed.shp'  # Change this
SAND_DUNE_SHAPEFILE_PATH = '/content/drive/My Drive/AP_Inputs/outputs/WithStructures/AP_SP_Filter_Merge_SandDune.shp'  # Change this
ARTIFICIAL_STRUCTURE_SHAPEFILE_PATH = '/content/drive/My Drive/AP_Inputs/outputs/WithStructures/AP_ArtificialStructure.shp'  # Change this

# Output path
OUTPUT_SHAPEFILE_PATH = '/content/drive/My Drive/AP_Inputs/outputs/WithStructures/CIRA_withstructures.shp'  # Change this

# Read shapefiles
print("Reading shapefiles...")
risk_gdf = gpd.read_file(RISK_SHAPEFILE_PATH)
sand_dune_gdf = gpd.read_file(SAND_DUNE_SHAPEFILE_PATH)
artificial_structure_gdf = gpd.read_file(ARTIFICIAL_STRUCTURE_SHAPEFILE_PATH)

print(f"Risk shapefile: {len(risk_gdf)} features")
print(f"Sand dune shapefile: {len(sand_dune_gdf)} features")
print(f"Artificial structure shapefile: {len(artificial_structure_gdf)} features")
print(f"Risk shapefile columns: {risk_gdf.columns.tolist()}")

# Ensure all GeoDataFrames have the same CRS
print(f"\nRisk CRS: {risk_gdf.crs}")
print(f"Sand dune CRS: {sand_dune_gdf.crs}")
print(f"Artificial structure CRS: {artificial_structure_gdf.crs}")

# Reproject if needed to match risk_gdf CRS
if sand_dune_gdf.crs != risk_gdf.crs:
    sand_dune_gdf = sand_dune_gdf.to_crs(risk_gdf.crs)
    print("Sand dune shapefile reprojected to match risk CRS")

if artificial_structure_gdf.crs != risk_gdf.crs:
    artificial_structure_gdf = artificial_structure_gdf.to_crs(risk_gdf.crs)
    print("Artificial structure shapefile reprojected to match risk CRS")

# Union all sand dunes and artificial structures into single geometry
print("\nUnioning sand dunes and artificial structures...")

# Attempt to fix invalid geometries with buffer(0) before unioning
sand_dune_gdf['geometry'] = sand_dune_gdf.geometry.buffer(0)
artificial_structure_gdf['geometry'] = artificial_structure_gdf.geometry.buffer(0)

sand_dune_union = unary_union(sand_dune_gdf.geometry)
artificial_structure_union = unary_union(artificial_structure_gdf.geometry)
combined_features_union = unary_union([sand_dune_union, artificial_structure_union])

# Function to reclassify risk
def reclassify_risk(risk_class, intersects_features):
    """
    Reclassifies risk based on whether it intersects with sand dunes or artificial structures

    Rules:
    - High risk → Medium risk (if intersects)
    - Medium risk → Low risk (if intersects)
    - Low remains Low
    """
    if not intersects_features:
        return risk_class

    if risk_class.upper() == 'HIGH':
        return 'MEDIUM'
    elif risk_class.upper() == 'MEDIUM':
        return 'LOW'
    elif risk_class.upper() == 'LOW':
        return 'LOW'
    else:
        return risk_class

# Identify the risk class column (adjust if your column name is different)
# Common column names: 'class', 'risk_class', 'risk_rank', 'RISK', etc.
risk_column = None
possible_columns = ['risk_rank', 'class', 'risk_class', 'RISK', 'risk', 'Risk', 'CLASS', 'Class']

for col in possible_columns:
    if col in risk_gdf.columns:
        risk_column = col
        break

if risk_column is None:
    print("Available columns:", risk_gdf.columns.tolist())
    risk_column = input("Enter the name of the risk classification column: ")

print(f"Using column '{risk_column}' for risk classification")
print(f"Unique values in {risk_column}: {risk_gdf[risk_column].unique()}")

# Apply reclassification
print("\nApplying reclassification logic...")
risk_gdf['original_risk'] = risk_gdf[risk_column]  # Keep original for reference

# Check intersection with combined features
risk_gdf['intersects_features'] = risk_gdf.geometry.intersects(combined_features_union)

# Apply reclassification function
risk_gdf[risk_column] = risk_gdf.apply(
    lambda row: reclassify_risk(row[risk_column], row['intersects_features']),
    axis=1
)

# Show changes
print("\nReclassification Summary:")
changes = pd.DataFrame({
    'Original Risk': risk_gdf['original_risk'],
    'New Risk': risk_gdf[risk_column],
    'Intersects Features': risk_gdf['intersects_features']
})

print(changes[changes['Original Risk'] != changes['New Risk']])
print(f"\nTotal features affected: {(changes['Original Risk'] != changes['New Risk']).sum()}")

# Remove temporary columns if desired (keep original_risk for verification)
# risk_gdf = risk_gdf.drop(columns=['intersects_features'])

# Save the reclassified shapefile
print(f"\nSaving reclassified shapefile to: {OUTPUT_SHAPEFILE_PATH}")
os.makedirs(os.path.dirname(OUTPUT_SHAPEFILE_PATH), exist_ok=True)
risk_gdf.to_file(OUTPUT_SHAPEFILE_PATH)

print("\u2713 Shapefile successfully saved!")
print("\nFinal risk distribution:")
print(risk_gdf[risk_column].value_counts())

# Display sample of changes
print("\nSample of reclassified features (first 10 changes):")
changed_features = risk_gdf[risk_gdf['original_risk'] != risk_gdf[risk_column]][['original_risk', risk_column, 'intersects_features']].head(10)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading shapefiles...
Risk shapefile: 706860 features
Sand dune shapefile: 22537 features
Artificial structure shapefile: 134 features
Risk shapefile columns: ['elevation_', 'distance_b', 'risk_level', 'risk_code', 'area_m2', 'area_km2', 'geometry']

Risk CRS: EPSG:32644
Sand dune CRS: EPSG:32644
Artificial structure CRS: EPSG:4326
Artificial structure shapefile reprojected to match risk CRS

Unioning sand dunes and artificial structures...
Available columns: ['elevation_', 'distance_b', 'risk_level', 'risk_code', 'area_m2', 'area_km2', 'geometry']
Enter the name of the risk classification column: risk_level
Using column 'risk_level' for risk classification
Unique values in risk_level: ['High' 'Medium' 'Low']

Applying reclassification logic...

Reclassification Summary:
       Original Risk New Risk  Intersects Features
2             Medium      LOW         

/tmp/ipykernel_860/4127283169.py:131: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  risk_gdf.to_file(OUTPUT_SHAPEFILE_PATH)
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'original_risk' to 'original_r'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'intersects_features' to 'intersects'
  ogr_write(


✓ Shapefile successfully saved!

Final risk distribution:
risk_level
Low       501941
Medium    102278
High       63892
LOW        38379
MEDIUM       370
Name: count, dtype: int64

Sample of reclassified features (first 10 changes):


In [4]:
import os

# Set the configuration option to allow geopandas to restore or create the .shx file
os.environ['SHAPE_RESTORE_SHX'] = 'YES'

print("Set SHAPE_RESTORE_SHX config option to YES. Please re-run the previous cell.")

Set SHAPE_RESTORE_SHX config option to YES. Please re-run the previous cell.
